# Sri Lanka Leptospirosis: ensemble research

Precise feature decisions, comparisons of the approved classification models, development-only model selection, internal holdout evaluation, and explanations of the selected predictor.

Patient data and generated models/results are not distributed. Obtain the dataset from the source cited in README.md. This notebook reads your locally generated experiment. Source outcome meanings, units and the 99 missing convention remain provisional.

## 1. Configure and reproduce the experiment
Use the Python 3.12 environment from `requirements-dev.txt`. Train first with `python train.py --data /path/to/file.csv`, or set `DATA_PATH` and enable `RUN_TRAINING` below. All outputs remain in the ignored local output directory.

The setup cell can find the project from the notebook folder, repository root, `ML Assignment`, or `Applied ML`. In VS Code, select this repository's `.venv/Scripts/python.exe` as the notebook kernel, then restart the kernel and run all cells. For an unrelated working directory, set `PROJECT_DIR` explicitly in the setup cell.

In [1]:
!pip install pandas numpy -q

In [2]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image
# Set PROJECT_DIR explicitly only if your kernel starts outside this project/workspace.
PROJECT_DIR = None
start = Path(PROJECT_DIR).expanduser().resolve() if PROJECT_DIR else Path.cwd().resolve()
project_name = 'sri-lanka-leptospirosis-ensemble-ml'
candidates = []
for folder in [start, *start.parents]:
    candidates.extend([folder, folder / project_name, folder / 'ML Assignment' / project_name])
ROOT = next((folder for folder in candidates
             if (folder / 'train.py').is_file()
             and (folder / 'src/data.py').is_file()
             and (folder / 'notebooks/modelling.ipynb').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Cannot find the research project. Set PROJECT_DIR above to the folder containing train.py and notebooks/.')
print('Project folder:', ROOT)
print('Python environment:', sys.executable)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
DATA_PATH = ROOT / 'data' / 'Leptospirosis clinical data.csv'  # Or an external local path.
OUTPUT_DIR = ROOT / 'outputs'
RUN_TRAINING = False
REPORTS = OUTPUT_DIR / 'reports'


Project folder: D:\Masters\Second Semester\Applied ML\ML Assignment\sri-lanka-leptospirosis-ensemble-ml
Python environment: d:\Masters\Second Semester\Applied ML\ML Assignment\.venv\Scripts\python.exe


In [ ]:
if RUN_TRAINING:
    from train import run
    run(data_path=DATA_PATH, output=OUTPUT_DIR)
if not (OUTPUT_DIR / 'artifacts/manifest.json').is_file():
    raise FileNotFoundError('Train locally first using python train.py --data /path/to/file.csv, or set DATA_PATH and RUN_TRAINING above.')
read_json = lambda name: json.loads((REPORTS / name).read_text(encoding='utf-8'))
selection = read_json('selection.json')
test = read_json('test_results.json')
cv = pd.read_csv(REPORTS / 'cv_candidates.csv')
folds = pd.read_csv(REPORTS / 'cv_folds.csv')
leaderboard = pd.read_csv(REPORTS / 'cv_leaderboard.csv')
from src.models import candidates as declared_candidates
if set(cv['model']) != {spec['model'] for spec in declared_candidates()}:
    raise ValueError('Existing results use a different model scope. Run full training for the approved models first.')
from src.runtime import load_release
model, reference, manifest = load_release(OUTPUT_DIR)
print('Verified model version:', manifest['model_version'])
print(test['notice'])


## 2. Feature engineering with precise reasoning
The decisions below state the exact operation, reason, assumptions, leakage boundary and validation evidence. Derived fields are computed before imputation. Learned preprocessing is fitted only on fitting folds.

In [ ]:
display(Markdown((ROOT / 'docs/FEATURE_ENGINEERING.md').read_text(encoding='utf-8')))
display(pd.read_csv(REPORTS / 'feature_audit.csv'))

### Inspect the actual transform and synthetic example
The transformer does not learn population statistics. Inspecting its source makes the notebook consistent with the training and serving implementation.

In [ ]:
import inspect
from src.data import AdmissionFeatures, validate_record
from src.models import pipeline
print(inspect.getsource(AdmissionFeatures))
print(inspect.getsource(pipeline))
synthetic = validate_record({'Age':35, 'Feverad':1, 'SBPadd':120, 'DBPadd':80})
display(AdmissionFeatures(True).fit_transform(synthetic)[['Age','pulse_pressure','mean_arterial_pressure','symptoms_recorded_fraction','exposures_recorded_fraction']])

### Development-only ablation
For each method, compare its best base configuration with its best engineered configuration. The full candidate table allows matched-hyperparameter comparisons. A positive difference supports that representation within this search; it is not proof of significance or a causal feature effect.

In [ ]:
display(pd.read_csv(REPORTS / 'feature_ablation.csv'))

## 3. Shared evaluation and declared recipes
An 80/20 stratified seed-42 split separates development from internal holdout records. Actual cohort/split counts are recorded in your generated test_results.json. Every recipe uses the same five development folds. The 36 recipes cover 10 approved classification methods with equal class weights. Linear Regression is approved for continuous-target tasks but is not fitted to this binary target. This historical holdout was already used in the earlier project, so it is not fresh external validation.

Within each outer fold, the model owns the feature transformer, imputer, encoder and scaler. Only Logistic Regression, Decision Tree, Random Forest, SVM, KNN, XGBoost, AdaBoost, Gradient Boosting, LightGBM and CatBoost enter selection. No additional ensemble combinations are used.

In [ ]:
from src.models import candidates
recipes = candidates()
display(pd.DataFrame([{k:v for k,v in spec.items() if k != 'estimator'} for spec in recipes]))
print('Candidates:', len(recipes), '| outer fold rows:', len(folds))

## 4. Decision Tree and Random Forest
A single **Decision Tree** learns split rules; **Random Forest** averages bootstrap-fitted trees with random feature subsets at splits. Depth and leaf-size constraints control capacity. Each row is the method's best development recipe; variability is fold SD, not a confidence interval.

In [ ]:
display(leaderboard.loc[leaderboard.family.isin(['Tree', 'Bagging']), ['model','feature_set','mean_roc_auc','std_roc_auc','mean_average_precision','mean_f1','mean_recall','mean_brier','mean_train_roc_auc','mean_fit_predict_seconds']])

## 5. Boosting
Compare **AdaBoost**, **gradient boosting**, **XGBoost**, **LightGBM** and **CatBoost**. Sequential learners improve a fitting objective; learning rate, shallow trees and regularization constrain capacity. All receive the same prepared features. CatBoost therefore uses one-hot inputs here; its native categorical mode is not evaluated. The compact grids are explicit and do not establish each library's best possible configuration.

In [ ]:
display(leaderboard.loc[leaderboard.family == 'Boosting', ['model','feature_set','mean_roc_auc','std_roc_auc','mean_average_precision','mean_f1','mean_recall','mean_brier','mean_train_roc_auc','mean_fit_predict_seconds']])

## 6. Logistic Regression, SVM and KNN
**Logistic Regression** models the log odds of the binary label. **SVM** uses an RBF kernel, with probability estimation enabled for the common evaluation interface. **KNN** averages labels of nearby prepared records. All use the same fold-fitted imputation, encoding and numeric scaling. Their compact development grids compare regularization strength or neighbor count. Linear Regression remains in the approved list for regression tasks; this experiment has a binary target.

In [ ]:
display(leaderboard.loc[leaderboard.family.isin(['Linear', 'Kernel', 'Neighbors']), ['model','feature_set','mean_roc_auc','std_roc_auc','mean_average_precision','mean_f1','mean_recall','mean_brier','mean_train_roc_auc','mean_fit_predict_seconds']])

## 7. Select the complete recipe using explicit metrics
**Primary:** maximize mean development ROC AUC. **Exact ties:** maximize mean development average precision, then deterministic name/feature/candidate order. ROC AUC measures ranking across thresholds; AP provides prevalence-sensitive precision/recall context. Threshold-dependent F1, recall, precision and balanced accuracy describe behaviour at 0.50. Brier score and log loss describe probability error; lower is better. Training AUC and timing expose overfitting/computational cost but do not choose the winner.

The saved selection precedes holdout prediction. We do not pick the model by accuracy, by its test AUC, or by changing metrics after seeing results. Fold SD does not establish significance; selecting from many recipes can make the winning CV estimate optimistic.

In [ ]:
from train import ranked
ranked_cv = ranked(cv)
display(ranked_cv[['model','family','feature_set','candidate','mean_roc_auc','std_roc_auc','mean_average_precision']])
display(selection)
assert int(ranked_cv.iloc[0].spec_id) == selection['spec_id']
assert selection['test_used_for_selection'] is False
print('Chosen:', selection['selected_model'], '/', selection['feature_set'])
display(Image(filename=str(REPORTS / 'comparison.png')))

### Selection conclusion
The ranking above identifies the best complete recipe under the declared development criterion. Compare matched base/engineered configurations in `cv_candidates.csv` and the representation ablation before deciding whether added features earn their complexity. Fold variability and a winning mean do not establish statistical or universal superiority. The historical reference run in `docs/MODEL_CARD.md` is separate from your newly generated results.

## 8. Evaluate the frozen winner
Only the selected model is fitted for the new holdout report. The prior baseline is a constant development-prevalence reference, not a competing trained algorithm. All precision/recall/F1 values use source Final code 1 as positive and threshold 0.50. Bootstrap intervals describe the fixed fitted model; they exclude full-search/selection and prevalence uncertainty. The historical holdout is internal research evidence.

In [ ]:
display(pd.DataFrame([{'metric':k, 'selected_model':v, 'prior_baseline':test['baseline_metrics'][k],
    'lower_95_percent':test['confidence_intervals'][k][0], 'upper_95_percent':test['confidence_intervals'][k][1]} for k,v in test['metrics'].items()]))
display(pd.DataFrame(test['confusion_matrix'], index=['Actual code 2','Actual code 1'], columns=['Predicted code 2','Predicted code 1']))
display(Image(filename=str(REPORTS / 'evaluation.png')))
print(test['confidence_interval_method'])

### Interpretation
Read ROC AUC together with average precision and code-1 prevalence. Inspect false negatives and recall at the fixed 0.50 threshold, rather than treating above-chance ranking as adequate performance. The reliability plot is descriptive; outputs are uncalibrated. A future operating threshold needs verified outcome meaning and an explicit objective, chosen using development data and followed by independent evaluation.

## 9. Explain the chosen model after selection
### Global: original-feature permutation importance
Shuffle each original input on the internal holdout and measure the decrease in ROC AUC, repeating ten times. The full pipeline is rerun, preserving dependencies with any derived fields. Importance follows selection and does not drive post-test feature removal. Error bars show shuffle SD, not causal uncertainty. Correlated inputs can share/mask importance and negative values can occur.

In [ ]:
display(pd.read_csv(REPORTS / 'permutation_importance.csv'))
display(Image(filename=str(REPORTS / 'importance.png')))

### Local: reference-based Monte Carlo Shapley explanation
Use 24 random feature orders against one synthetic reference: development numeric medians and all source categories missing. Each order replaces reference values with example values and records marginal changes in code-1 probability. The average is a finite-permutation approximation; contributions telescope so baseline plus their sum reconstructs the prediction. This is a baseline-substitution game, not conditional population SHAP. Contributions depend on the reference, substitutions can be uncommon, and no causal or clinical interpretation follows.

In [ ]:
from src.explain import local_shapley
example = validate_record({'Age':35, 'Feverad':1})
base, predicted, local = local_shapley(model, example, reference)
print(f'Baseline {base:.6f} + sum {local.contribution.sum():+.6f} = prediction {predicted:.6f}')
assert np.isclose(base + local.contribution.sum(), model.predict_proba(example)[0,1])
display(local)
ax = local.head(12).iloc[::-1].plot.barh(x='feature', y='contribution', color='#087f8c', legend=False, title='Synthetic example: probability contributions')
ax.set_xlabel('Change relative to synthetic reference')
import matplotlib.pyplot as plt
plt.show()

### Synthetic one-feature response
The curve varies age with Feverad=1, other source categories missing and other numerics at development medians. This one fixed synthetic context is not a population effect and exposes no actual patient row.

In [ ]:
response = pd.read_csv(REPORTS / 'synthetic_age_response.csv')
response.plot(x='Age', y='code1_probability', title='Synthetic age-response context', legend=False)
plt.show()

## 10. Reproducibility and research extensions
The local manifest records data/code/package/report hashes, seed and selected representation. Raw data, fitted models and generated outputs are excluded from version control. The loader verifies model/report bytes and package compatibility; load only your own trusted generated artifacts.

To extend the experiment, declare new recipes in `src/models.py` or change the feature contract in `src/data.py`. Preserve fold-local preprocessing and the approved model scope, use a new output directory, and obtain independent validation instead of repeatedly tuning against the historical holdout.

In [ ]:
display({k:manifest[k] for k in ['model_version','model','feature_set','created_utc','python','source_data_sha256','packages']})

## 11. Selected model card and limitations
The model card below is generated from the current approved-model run. The archived original experiment in docs/MODEL_CARD.md is superseded and is not part of this comparison.

In [ ]:
display(Markdown((REPORTS / 'MODEL_CARD.md').read_text(encoding='utf-8')))

## Sources
- Suneth Agampodi (2022), [Mendeley Data v4](https://data.mendeley.com/datasets/zmxmp42g9p/4), DOI 10.17632/zmxmp42g9p.4, CC BY 4.0. Obtain data directly from the publisher; no data is redistributed here.
- [scikit-learn permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html): score-based model inspection and limitations.